# AI Regime + Scenarios (5000) — chạy detect rồi sinh kịch bản tier "final"

Notebook này gọi lại đúng 2 hàm CLI thật của `packages/ai`
(`qshield_ai.cli.regime`, `qshield_ai.cli.scenarios`) — không chứa logic tài chính/thống kê riêng,
đúng quy tắc CLAUDE.md ("notebook không được chứa core logic — chỉ gọi lại hàm trong `packages/`").

**Luồng:** Detect (HMM regime) → Scenarios với `num_scenarios=5000` (tier "final" theo
`configs/profiles/workflow_update.yaml`, khác với mặc định `500` đang khóa trong
`configs/scenarios.yaml` cho tier dev/demo).

**`artifacts.mode: dev`** — ghi vào đường dẫn cố định `artifacts/dev/regime/` và
`artifacts/dev/scenarios/` (không tạo thư mục riêng — đúng CLAUDE.md quy tắc 8: mọi đường dẫn
artifact chỉ sinh từ `qshield_contracts.paths.ArtifactPaths`). Mỗi lần chạy lại sẽ **đè lên** kết
quả trước đó ở dev mode; nếu sau này cần version theo `run_id` (tier bằng chứng chính thức), đổi
`artifacts.mode: runs` trong `configs/base.yaml` trước khi chạy — không cần sửa gì trong notebook
này.

**Điều kiện tiên quyết:** đã chạy xong `notebooks/exploration/data_exploration.ipynb` (hoặc
`uv run qshield-data build`) — notebook này KHÔNG tự fetch/clean data, chỉ đọc
`data/processed/{returns,market_features}.parquet` đã có sẵn.

Muốn chạy tương đương ngoài notebook: `uv run qshield-ai regime --config configs/base.yaml` rồi
`uv run qshield-ai scenarios --config configs/base.yaml` (nhớ đổi `num_scenarios` trong
`configs/scenarios.yaml` thành `5000` trước, hoặc dùng file config đã resolve mà notebook này ghi
ra — xem Bước 0).

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
import yaml
from qshield_contracts.config import Config


def _find_project_root(marker: str = "CLAUDE.md") -> Path:
    # Dò lên theo marker (không đếm cứng số cấp) — an toàn khi chạy lại cell này lần 2 trong cùng
    # kernel, lúc cwd đã đổi thành PROJECT_ROOT từ os.chdir() bên dưới (cùng lý do đã áp dụng ở
    # notebooks/exploration/data_exploration.ipynb).
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(
        f"Không tìm thấy {marker} từ {p} trở lên — notebook phải nằm trong repo QSHIELD."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)

CONFIG_PATH = PROJECT_ROOT / "configs" / "base.yaml"
cfg = Config.load(CONFIG_PATH)
cfg["artifacts"]["mode"]

'dev'

> **Vì sao `os.chdir(PROJECT_ROOT)`:** giống hệt lý do ở `data_exploration.ipynb` — cwd mặc định
> của kernel Jupyter là thư mục chứa `.ipynb`, không phải gốc repo, trong khi `ArtifactPaths` build
> đường dẫn `artifacts/...` tương đối theo repo root. Nếu restart kernel, luôn chạy lại cell đầu
> tiên trước khi nhảy vào cell khác.

## Bước 0: Kiểm tra điều kiện tiên quyết + dựng config override (5000 kịch bản)

In [2]:
required_data_files = [
    PROJECT_ROOT / "data" / "processed" / "returns.parquet",
    PROJECT_ROOT / "data" / "processed" / "market_features.parquet",
]
missing = [p for p in required_data_files if not p.exists()]
if missing:
    raise RuntimeError(
        "Thiếu " + ", ".join(str(p) for p in missing) + " — chạy "
        "notebooks/exploration/data_exploration.ipynb (hoặc `uv run qshield-data build`) trước."
    )
print("OK — đủ file data đầu vào cho regime.")

OK — đủ file data đầu vào cho regime.


In [3]:
from qshield_contracts.paths import ArtifactPaths

# `num_scenarios: 500` trong configs/scenarios.yaml là mặc định tier dev/demo (đã khóa, dùng chung
# cho mọi người) — KHÔNG sửa trực tiếp file đó. Thay vào đó merge một bản config đã load (giữ mọi
# giá trị khác nguyên vẹn) rồi ghi ra file resolved riêng cho lần chạy 5000 kịch bản này, đúng kiểu
# `packages/pipeline/run_context.py` đã làm để chia sẻ config giữa các bước mà không đụng file gốc.
NUM_SCENARIOS_FINAL = 5000

resolved_cfg = dict(cfg)
resolved_cfg["num_scenarios"] = NUM_SCENARIOS_FINAL

paths = ArtifactPaths(
    resolved_cfg, run_id=None
)  # dev mode: run_id không ảnh hưởng đường dẫn
paths.run_root.mkdir(parents=True, exist_ok=True)
resolved_config_path = paths.run_root / "_ai_5000_resolved_config.yaml"
resolved_config_path.write_text(
    yaml.safe_dump(resolved_cfg, allow_unicode=True), encoding="utf-8"
)
print(
    f"Config đã resolve (num_scenarios={NUM_SCENARIOS_FINAL}) → {resolved_config_path}"
)

Config đã resolve (num_scenarios=5000) → artifacts/dev/_ai_5000_resolved_config.yaml


## Bước 1: Detect (Regime — HMM)

Gọi thẳng `qshield_ai.cli.regime()` — huấn luyện/suy luận Gaussian HMM trên
`market_features.parquet`, chọn seed champion theo stability medoid (không cherry-pick), gán nhãn
Normal/Volatile/Stress theo đặc trưng thống kê (CLAUDE.md quy tắc 7).

Output: `artifacts/dev/regime/regime_daily.parquet`, `regime_summary.json`, `regime_selection.csv`.

Bắt đầu tính giờ từ đây (`RUN_START`) — tổng thời gian Detect + Scenarios được in ở cuối notebook.

In [4]:
from qshield_ai.cli import regime as ai_regime

RUN_START = time.perf_counter()
ai_regime(config=str(resolved_config_path), mock=False)

Feature frame: 2571 dòng, 2016-04-04 → 2026-07-30


Champion seed=303, 2571 dòng regime đã ghi.


[regime] OK — 2571 dòng → artifacts/dev/regime


In [5]:
import json

regime_summary_path = (
    PROJECT_ROOT / "artifacts" / "dev" / "regime" / "regime_summary.json"
)
regime_summary = json.loads(regime_summary_path.read_text(encoding="utf-8"))
{
    "run_mode": regime_summary.get("run_mode"),
    "gate_status": regime_summary.get("gate_status"),
    "gate_reasons": regime_summary.get("gate_reasons"),
    "champion_seed": regime_summary.get("champion", {}).get("seed"),
}

{'run_mode': 'NON_BASELINE_RUN',
 'gate_status': 'OK',
 'gate_reasons': [],
 'champion_seed': 303}

**Nếu `gate_status` không phải trạng thái pass** (đọc đúng field/giá trị thật trong
`regime_summary.json` ở trên) — dừng lại, đừng chạy tiếp Bước 2. Đọc
`docs/runbook/troubleshooting.md` để biết cách xử lý (thường là HMM không hội tụ ổn định qua các
seed → rơi về `rule_based_regime`, kết quả yếu hơn và phải công bố rõ).

## Bước 2: Scenarios (5000 kịch bản, tier "final")

Gọi thẳng `qshield_ai.cli.scenarios()` trên CHÍNH config đã resolve ở Bước 0 (`num_scenarios=5000`)
— regime-conditioned moving-block bootstrap, block 5 ngày × 4 = 20 ngày horizon.

`force=False`: nếu scenario validation gate FAIL hoặc regime đang ở fallback rule-based, lệnh sẽ
DỪNG thay vì âm thầm ghi cube — đúng vì đây là chạy cho bằng chứng, không phải debug nhanh. Muốn ép
ghi dù gate FAIL (đánh dấu `NON_BASELINE_RUN`), đổi `force=True` bên dưới, nhưng phải biết rõ lý do
tại sao trước khi làm vậy.

Output: `artifacts/dev/scenarios/stress_scenarios.npz` (shape `(5000, 20, 8)`),
`scenario_manifest.json`, `reports/scenario_validation.csv`... (đường dẫn thật xem log của lệnh).

In [6]:
from qshield_ai.cli import scenarios as ai_scenarios

ai_scenarios(config=str(resolved_config_path), mock=False, force=False)

total_seconds = time.perf_counter() - RUN_START
print(
    f"[timing] Detect + Scenarios ({NUM_SCENARIOS_FINAL} kịch bản): {total_seconds:.1f}s"
)

Ngày đánh giá t=2026-07-30, regime mục tiêu=volatile


[scenarios] PASS — cube (5000, 20, 8) regime=volatile t=2026-07-30 → artifacts/dev/scenarios
[timing] Detect + Scenarios (5000 kịch bản): 22.5s


In [7]:
validation_path = (
    PROJECT_ROOT / "artifacts" / "dev" / "scenarios" / "scenario_validation.csv"
)
validation = pd.read_csv(validation_path)
n_fail = int((validation["verdict"] == "FAIL").sum())
print(f"{len(validation)} metric — {n_fail} FAIL")
validation[validation["verdict"] != "PASS"] if n_fail else validation

27 metric — 0 FAIL


,target_regime,reference_windows,small_sample,metric,scenario_value,reference_value,statistic,threshold_low,threshold_high,verdict,note
0,normal,569,False,mean_abs_diff,0.001406,0.001577,0.000171,NaN,0.001,PASS,NaN
1,normal,569,False,std_ratio,0.015812,0.016326,0.968464,0.8,1.250,PASS,NaN
2,normal,569,False,skew_abs_diff,0.481059,0.457431,0.023628,NaN,1.000,PASS,NaN
3,normal,569,False,kurtosis_abs_diff,3.403058,3.308500,0.094559,NaN,5.000,PASS,NaN
4,normal,569,False,q05_rel_error,-0.022027,-0.022685,0.029004,NaN,0.250,PASS,NaN
5,normal,569,False,q95_rel_error,0.028472,0.029742,0.042708,NaN,0.250,PASS,NaN
6,normal,569,False,autocorr_abs_diff,0.017358,0.012481,0.004877,NaN,0.100,PASS,NaN
7,normal,569,False,corr_mean_abs_diff,0.236017,0.251336,0.015320,NaN,0.100,PASS,NaN
8,normal,569,False,tail_coverage_ratio,0.043950,0.029340,1.497934,0.5,2.000,PASS,NaN
9,volatile,834,False,mean_abs_diff,0.000275,0.000164,0.000110,NaN,0.001,PASS,NaN


## Xong — checklist đầu ra

- `artifacts/dev/regime/{regime_daily.parquet,regime_summary.json,regime_selection.csv}`
- `artifacts/dev/scenarios/{stress_scenarios.npz,scenario_manifest.json,scenario_validation.csv}`
- `artifacts/dev/{config.json,data_version.json,metrics.json,logs.txt}` — do `RunContext` ghi
  (CLAUDE.md quy tắc 13), dùng chung `run_root` với cả hai bước ở trên vì cùng `artifacts.mode: dev`.
- `artifacts/dev/_ai_5000_resolved_config.yaml` — bản config đã dùng thật cho lần chạy này (kèm
  `num_scenarios: 5000`), giữ lại để đối chiếu/tái lập, không phải artifact chuẩn theo schema.

**Muốn chạy lại từ đầu:** chỉ cần chạy lại từ Cell 1 (không cần restart kernel) — `dev` mode ghi đè
đường dẫn cố định nên không tích tụ file cũ. Muốn giữ version theo từng lần chạy (không bị đè), đổi
`artifacts.mode: runs` trong `configs/base.yaml` trước khi chạy lại notebook này.